# 00b — Exploratory Data Analysis & Statistics

**EDA is not just running `.describe()`.** It is a structured investigation that shapes every modeling decision that follows. This notebook trains you to do EDA the way a senior analyst would: systematically, with business questions driving each step.

**Topics:** Distribution analysis, correlation matrices, skewness/kurtosis, crosstabs, univariate/bivariate analysis, hypothesis testing foundations, statistical summaries.

**Reference:** [pandas docs](https://pandas.pydata.org/docs/) | [numpy docs](https://numpy.org/doc/stable/) | [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)

**Allowed:** `pandas`, `numpy`, `scipy.stats`

**Dataset:** E-commerce transactions — customer orders, products, returns, revenue.


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from sklearn.datasets import fetch_openml

# Online Retail II — richer version with returns
retail_raw = fetch_openml(name='onlineretail', version=1, as_frame=True, parser='auto').frame
retail = retail_raw.copy()
retail.columns = [c.strip() for c in retail.columns]
retail['InvoiceDate'] = pd.to_datetime(retail['InvoiceDate'])
retail['Quantity'] = pd.to_numeric(retail['Quantity'], errors='coerce')
retail['UnitPrice'] = pd.to_numeric(retail['UnitPrice'], errors='coerce')
retail = retail.dropna(subset=['CustomerID'])
retail['CustomerID'] = retail['CustomerID'].astype(int)
retail['Revenue'] = retail['Quantity'] * retail['UnitPrice']
retail['IsReturn'] = retail['Quantity'] < 0
retail['Month'] = retail['InvoiceDate'].dt.to_period('M')
retail['DayOfWeek'] = retail['InvoiceDate'].dt.day_name()
retail['Hour'] = retail['InvoiceDate'].dt.hour

# Forward-only transactions for most exercises
orders = retail[retail['Quantity'] > 0].copy()
print(f"Full dataset: {retail.shape}")
print(f"Orders only: {orders.shape}")
orders.head()

---
## Exercise 1 — Descriptive Statistics Beyond `.describe()`

`.describe()` gives you 8 numbers. Real EDA needs more.

Write `extended_describe(df, numeric_cols)` that returns a DataFrame with one row per column and these statistics:
- `mean`, `median`, `std`, `min`, `max`
- `skewness`: Fisher's definition (use `scipy.stats.skew` or derive manually)
- `kurtosis`: excess kurtosis (normal = 0)
- `iqr`: Q75 - Q25
- `cv`: coefficient of variation = std / mean (as %)
- `pct_zero`: % of values that are exactly 0
- `pct_negative`: % of values below 0

All values rounded to 4dp. This function must work on any DataFrame.

In [ ]:
def extended_describe(df: pd.DataFrame, numeric_cols: list) -> pd.DataFrame:
    """
    Extended descriptive statistics for numeric columns.
    Returns DataFrame indexed by column name.
    """
    # YOUR CODE HERE
    pass

numeric_cols = ['Quantity', 'UnitPrice', 'Revenue']
desc = extended_describe(orders, numeric_cols)

In [ ]:
# --- ASSERTIONS ---
expected_cols = ['mean', 'median', 'std', 'min', 'max', 'skewness',
                 'kurtosis', 'iqr', 'cv', 'pct_zero', 'pct_negative']
assert list(desc.columns) == expected_cols, f"Columns mismatch: {list(desc.columns)}"
assert list(desc.index) == numeric_cols
assert desc['mean'].gt(0).all(), "All means should be positive for orders"
assert desc.loc['Revenue', 'skewness'] > 1, "Revenue is typically highly right-skewed"
assert (desc['pct_zero'] >= 0).all() and (desc['pct_zero'] <= 100).all()
assert (desc['pct_negative'] == 0).all(), "Orders only — no negatives"
print("✓ Exercise 1 passed")
print(desc)

---
## Exercise 2 — Distribution Analysis & Normality Testing

**Business question:** Before applying any parametric statistical test, you need to know if your data is normally distributed.

1. Write `test_normality(series, alpha=0.05)` that runs **three** normality tests:
   - Shapiro-Wilk (`scipy.stats.shapiro`) — use a random sample of 500 if series is larger
   - D'Agostino-Pearson (`scipy.stats.normaltest`)
   - Kolmogorov-Smirnov vs normal (`scipy.stats.kstest` with fitted mean/std)
   - Returns a dict: `{test_name: {'statistic': float, 'p_value': float, 'is_normal': bool}}`

2. Apply to `Revenue`, `UnitPrice`, `Quantity` from `orders`.

3. For each column, apply `np.log1p` transformation and re-test. Return `normality_comparison`: DataFrame showing `is_normal` before and after log transform for each test.

In [ ]:
def test_normality(series: pd.Series, alpha: float = 0.05) -> dict:
    """
    Runs 3 normality tests. Returns dict of results.
    Uses sample of 500 for Shapiro-Wilk if series > 500.
    """
    # YOUR CODE HERE
    pass

def normality_comparison_table(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """
    Returns DataFrame comparing normality before/after log1p.
    Columns: column, test, is_normal_raw, is_normal_log
    """
    # YOUR CODE HERE
    pass

normality_comparison = normality_comparison_table(orders, numeric_cols)

In [ ]:
# --- ASSERTIONS ---
result = test_normality(orders['Revenue'])
assert set(result.keys()) == {'shapiro', 'dagostino', 'ks'}
for test_name, res in result.items():
    assert set(res.keys()) == {'statistic', 'p_value', 'is_normal'}
    assert isinstance(res['is_normal'], bool)
# Revenue is highly skewed — should not be normal
assert not result['dagostino']['is_normal'], "Revenue should fail normality test"

assert list(normality_comparison.columns) == ['column', 'test', 'is_normal_raw', 'is_normal_log']
assert len(normality_comparison) == len(numeric_cols) * 3
print("✓ Exercise 2 passed")
print(normality_comparison)

---
## Exercise 3 — Correlation Analysis

**Business question:** Which features move together? Are there multicollinearity concerns?

First, build a customer-level aggregated dataset:

1. Aggregate `orders` by `CustomerID`:
   - `total_orders`: unique invoice count
   - `total_revenue`: sum of Revenue
   - `avg_order_value`: mean revenue per invoice
   - `total_items`: sum of Quantity
   - `avg_unit_price`: mean UnitPrice
   - `unique_products`: count of unique StockCode
   - `active_months`: count of unique months
   - `return_rate`: proportion of all their transactions that are returns (use full `retail` df)

2. Compute **Pearson**, **Spearman**, and **Kendall** correlation matrices on the customer features.

3. Write `correlation_report(df, method, threshold=0.7)` that returns pairs of features with `|correlation| >= threshold`. Columns: `feature_1`, `feature_2`, `correlation`. Sorted by `abs(correlation)` descending. No self-pairs, no duplicates.

In [ ]:
def build_customer_features(orders_df: pd.DataFrame, full_df: pd.DataFrame) -> pd.DataFrame:
    """
    Returns customer-level aggregated features DataFrame.
    CustomerID as index.
    """
    # YOUR CODE HERE
    pass

def correlation_report(df: pd.DataFrame, method: str = 'pearson',
                        threshold: float = 0.7) -> pd.DataFrame:
    """
    High-correlation pairs above threshold.
    Columns: feature_1, feature_2, correlation
    """
    # YOUR CODE HERE
    pass

customer_features = build_customer_features(orders, retail)
high_corr_pairs = correlation_report(customer_features, method='pearson', threshold=0.7)

In [ ]:
# --- ASSERTIONS ---
expected_feat_cols = ['total_orders', 'total_revenue', 'avg_order_value',
                      'total_items', 'avg_unit_price', 'unique_products',
                      'active_months', 'return_rate']
for col in expected_feat_cols:
    assert col in customer_features.columns, f"Missing: {col}"
assert customer_features.index.name == 'CustomerID'
assert customer_features['return_rate'].between(0, 1).all()

assert list(high_corr_pairs.columns) == ['feature_1', 'feature_2', 'correlation']
# No self-pairs
assert (high_corr_pairs['feature_1'] != high_corr_pairs['feature_2']).all()
# No duplicates (A,B and B,A)
pairs = set(tuple(sorted([r['feature_1'], r['feature_2']]))
            for _, r in high_corr_pairs.iterrows())
assert len(pairs) == len(high_corr_pairs), "Duplicate pairs found"
assert high_corr_pairs['correlation'].abs().is_monotonic_decreasing
print(f"✓ Exercise 3 passed — {len(high_corr_pairs)} high-correlation pairs")
print(high_corr_pairs)

---
## Exercise 4 — Bivariate Analysis & Crosstabs

**Business question:** How do categorical variables relate to each other and to the numeric target?

1. Build `revenue_by_dow`: mean and median revenue per `DayOfWeek`. Sort by mean revenue descending. Include `n_transactions`.

2. Build a **normalized crosstab** of `DayOfWeek` × `Country` (top 5 countries only) — cells = % of that country's orders falling on each day. Round to 2dp.

3. Write `group_comparison(df, group_col, value_col)` that:
   - Computes mean, median, std, and count of `value_col` per group.
   - Adds `pct_of_total`: each group's sum as % of the grand total.
   - Adds `index_vs_mean`: mean of group / overall mean * 100 (100 = average). This is a standard consulting metric.
   - Returns sorted by `index_vs_mean` descending.

4. Apply `group_comparison` to `Country` vs `Revenue` (top 10 countries by order count).

In [ ]:
def build_revenue_by_dow(df: pd.DataFrame) -> pd.DataFrame:
    """
    Revenue stats by day of week. Sorted by mean_revenue descending.
    Columns: DayOfWeek, mean_revenue, median_revenue, n_transactions
    """
    # YOUR CODE HERE
    pass

def group_comparison(df: pd.DataFrame, group_col: str,
                     value_col: str) -> pd.DataFrame:
    """
    Group-level summary with index_vs_mean.
    Returns sorted by index_vs_mean descending.
    """
    # YOUR CODE HERE
    pass

revenue_by_dow = build_revenue_by_dow(orders)

top10_countries = orders['Country'].value_counts().nlargest(10).index
country_comparison = group_comparison(
    orders[orders['Country'].isin(top10_countries)], 'Country', 'Revenue'
)

In [ ]:
# --- ASSERTIONS ---
assert list(revenue_by_dow.columns) == ['DayOfWeek', 'mean_revenue', 'median_revenue', 'n_transactions']
assert revenue_by_dow['mean_revenue'].is_monotonic_decreasing
assert len(revenue_by_dow) == 7, "Must have 7 days"

for col in ['mean', 'median', 'std', 'count', 'pct_of_total', 'index_vs_mean']:
    assert col in country_comparison.columns, f"Missing: {col}"
assert country_comparison['index_vs_mean'].is_monotonic_decreasing
assert abs(country_comparison['pct_of_total'].sum() - 100) < 0.1
# Index = 100 means average; top group should be above 100
assert country_comparison['index_vs_mean'].iloc[0] > 100
print("✓ Exercise 4 passed")
print(revenue_by_dow)
print(country_comparison)

---
## Exercise 5 — Percentile & Distribution Deep Dive

**Business question:** Revenue follows a power law — a small number of customers drive most revenue. Quantify this.

1. Compute the **Lorenz curve** for `total_revenue` in `customer_features`:
   - Sort customers by revenue ascending.
   - Compute cumulative share of customers (x) and cumulative share of revenue (y).
   - Return as `lorenz_df`: columns `cum_customers_pct`, `cum_revenue_pct`.

2. Compute the **Gini coefficient** from the Lorenz curve: `1 - 2 * area_under_lorenz_curve`. Use numpy trapezoid integration (`np.trapz`).

3. Compute `revenue_concentration`: a DataFrame showing what % of total revenue is held by the top 1%, 5%, 10%, 20% of customers.

4. Compute `percentile_profile`: the 5th, 10th, 25th, 50th, 75th, 90th, 95th, 99th percentile of `total_revenue`.

In [ ]:
def compute_lorenz(series: pd.Series):
    """
    Returns (lorenz_df, gini_coefficient)
    lorenz_df columns: cum_customers_pct, cum_revenue_pct
    """
    # YOUR CODE HERE
    pass

def revenue_concentration(series: pd.Series, top_pcts: list) -> pd.DataFrame:
    """
    Returns DataFrame: top_pct, n_customers, revenue_share_pct
    """
    # YOUR CODE HERE
    pass

lorenz_df, gini = compute_lorenz(customer_features['total_revenue'])
concentration = revenue_concentration(customer_features['total_revenue'], [1, 5, 10, 20])
percentile_profile = None  # YOUR CODE HERE

In [ ]:
# --- ASSERTIONS ---
assert list(lorenz_df.columns) == ['cum_customers_pct', 'cum_revenue_pct']
assert abs(lorenz_df['cum_customers_pct'].iloc[-1] - 100) < 0.1, "Must end at 100%"
assert abs(lorenz_df['cum_revenue_pct'].iloc[-1] - 100) < 0.1
assert lorenz_df['cum_revenue_pct'].is_monotonic_increasing

assert 0 < gini < 1, "Gini must be between 0 and 1"
assert gini > 0.4, "Revenue is typically highly concentrated (Gini > 0.4)"

assert list(concentration.columns) == ['top_pct', 'n_customers', 'revenue_share_pct']
assert concentration['revenue_share_pct'].is_monotonic_increasing

assert isinstance(percentile_profile, (pd.Series, dict))
print(f"✓ Exercise 5 passed — Gini coefficient: {gini:.4f}")
print(concentration)
print(f"Percentiles: {percentile_profile}")

---
## Exercise 6 — Time Series EDA

**Business question:** Is there seasonality? Are there anomalous periods? Is the business growing?

1. Build `monthly_ts`: monthly aggregation of `Revenue`, `n_orders`, `n_customers`, `avg_order_value`. DatetimeIndex.

2. Compute **month-over-month growth rates** for Revenue.

3. Detect **anomalous months**: months where revenue deviates more than 2 standard deviations from the rolling 3-month mean. Return `anomaly_months` as a list of Period values.

4. Compute **autocorrelation** of monthly Revenue at lags 1 through 6. Return as `autocorr_df`: columns `lag`, `autocorrelation`. Use pandas `.autocorr(lag=k)` method.

5. Compute a simple **trend**: fit a linear regression of Revenue on a time index (0, 1, 2...) using `numpy.polyfit`. Return `trend_slope` (revenue change per month) and `trend_r2`.

In [ ]:
def build_monthly_ts(df: pd.DataFrame) -> pd.DataFrame:
    """
    Monthly aggregation: Revenue, n_orders, n_customers, avg_order_value.
    DatetimeIndex.
    """
    # YOUR CODE HERE
    pass

def detect_anomalous_months(monthly_revenue: pd.Series, window: int = 3,
                             n_std: float = 2.0) -> list:
    """
    Returns list of period/date values where revenue is anomalous.
    """
    # YOUR CODE HERE
    pass

def compute_trend(monthly_revenue: pd.Series):
    """
    Returns (trend_slope, trend_r2) using linear fit.
    """
    # YOUR CODE HERE
    pass

monthly_ts = build_monthly_ts(orders)
anomaly_months = detect_anomalous_months(monthly_ts['Revenue'])
trend_slope, trend_r2 = compute_trend(monthly_ts['Revenue'])

In [ ]:
# --- ASSERTIONS ---
for col in ['Revenue', 'n_orders', 'n_customers', 'avg_order_value']:
    assert col in monthly_ts.columns, f"Missing: {col}"
assert isinstance(monthly_ts.index, pd.DatetimeIndex), "Must have DatetimeIndex"

assert isinstance(anomaly_months, list)
assert len(anomaly_months) >= 0  # May be 0 if data is smooth

assert isinstance(trend_slope, float)
assert 0 <= trend_r2 <= 1

# Autocorr check
autocorr_df = pd.DataFrame({
    'lag': range(1, 7),
    'autocorrelation': [monthly_ts['Revenue'].autocorr(lag=k) for k in range(1, 7)]
})
assert list(autocorr_df.columns) == ['lag', 'autocorrelation']
assert autocorr_df['autocorrelation'].between(-1, 1).all()

print(f"✓ Exercise 6 passed")
print(f"Trend slope: ${trend_slope:,.0f}/month | R²: {trend_r2:.4f}")
print(f"Anomalous months: {anomaly_months}")
print(autocorr_df)

---
## Exercise 7 — Cohort Analysis

**Business question:** Do customers acquired in different months have different retention and revenue patterns? This is one of the most common analytical tasks in consulting and product analytics.

1. Assign each customer a `cohort_month`: the month of their **first** purchase.

2. For each `(cohort_month, order_month)` pair, compute:
   - `cohort_period`: months since cohort acquisition (0 = acquisition month).
   - `n_customers`: unique customers active.
   - `total_revenue`.

3. Build `retention_matrix`: pivot table where rows = `cohort_month`, columns = `cohort_period` (0–11), values = **retention rate** (% of cohort still active vs period 0). Shape: `(n_cohorts, 12)`.

4. Build `revenue_matrix`: same structure but values = average revenue per active customer.

5. Compute `cohort_summary`: for each cohort, `cohort_size`, `m1_retention` (period 1 retention %), `m3_retention`, `avg_ltv_3m` (total revenue in first 3 periods / cohort size).

In [ ]:
def build_cohort_analysis(df: pd.DataFrame):
    """
    Returns (retention_matrix, revenue_matrix, cohort_summary)
    """
    # YOUR CODE HERE
    pass

retention_matrix, revenue_matrix, cohort_summary = build_cohort_analysis(orders)

In [ ]:
# --- ASSERTIONS ---
assert retention_matrix.shape[1] <= 12, "Max 12 cohort periods"
assert retention_matrix.iloc[:, 0].eq(100).all(), "Period 0 retention must be 100%"
assert (retention_matrix.fillna(0) <= 100).all().all()
assert (retention_matrix.fillna(0) >= 0).all().all()

for col in ['cohort_size', 'm1_retention', 'm3_retention', 'avg_ltv_3m']:
    assert col in cohort_summary.columns, f"Missing: {col}"
assert (cohort_summary['m1_retention'] <= 100).all()
assert (cohort_summary['cohort_size'] > 0).all()

print(f"✓ Exercise 7 passed — {len(retention_matrix)} cohorts")
print("Retention matrix (first 5 cohorts, first 6 periods):")
print(retention_matrix.iloc[:5, :6].round(1))
print(cohort_summary.head())

---
## Exercise 8 — EDA Report Generator

**Task:** Consolidate all EDA steps into a single automated report function — the kind you would run on any new dataset before presenting findings to a client.

Write `auto_eda_report(df, target_col=None)` that returns a dict with:
- `'shape'`: (rows, cols)
- `'numeric_summary'`: extended_describe on all numeric columns
- `'categorical_summary'`: for each categorical, top 5 value counts + % coverage
- `'missing_summary'`: columns with nulls, count, pct
- `'high_correlations'`: pairs with |correlation| > 0.7 (Pearson)
- `'skewed_features'`: list of numeric cols with |skewness| > 1
- `'target_correlation'`: if `target_col` provided, correlation of all numeric cols with target, sorted by absolute value

Apply to `customer_features` with `target_col='total_revenue'`.

In [ ]:
def auto_eda_report(df: pd.DataFrame, target_col: str = None) -> dict:
    """
    Automated EDA report. Works on any DataFrame.
    """
    # YOUR CODE HERE
    pass

eda_report = auto_eda_report(customer_features, target_col='total_revenue')

In [ ]:
# --- ASSERTIONS ---
required_keys = {'shape', 'numeric_summary', 'categorical_summary',
                 'missing_summary', 'high_correlations',
                 'skewed_features', 'target_correlation'}
assert set(eda_report.keys()) == required_keys
assert eda_report['shape'] == customer_features.shape
assert isinstance(eda_report['skewed_features'], list)
assert 'total_revenue' in eda_report['skewed_features'], "Revenue should be highly skewed"
assert isinstance(eda_report['target_correlation'], pd.Series)
assert eda_report['target_correlation'].abs().is_monotonic_decreasing
# Target itself should not be in correlation list
assert 'total_revenue' not in eda_report['target_correlation'].index
print("✓ Exercise 8 passed — Auto EDA report generated")
print(f"Skewed features: {eda_report['skewed_features']}")
print(f"Top correlates with revenue:\n{eda_report['target_correlation'].head()}")